### This Notebooks Contains the Code For Learning Stock Embeddings
This approach was described in:
- The original preprint: [Stock Embeddings: Learning Distributed Representations for Financial Assets](https://arxiv.org/pdf/2202.08968.pdf)
- The more updated conference publication which can be found [here](https://www.mdpi.com/2673-4591/39/1/30/pdf?version=1695109017)

For any questions, reach out to me at dolphrian@gmail.com

In [1]:
from stock_embeddings import ReturnsData
from stock_embeddings import get_pointwise_tgt_context_sets

# Constants and configurations
PERIODS = [1, 5, 10]
TRAIN_PCT = 1
CONTEXT_SIZE = 32

tgt_context_sets = []
for period in PERIODS:
    print(f"Running for period {period}")
    data = ReturnsData(
        daily_returns_path="../Data/returns_df_611.csv",
        extras_path="../Data/historical_stocks.csv",
    )
    data.change_returns_period(period)
    data.train_test_split(TRAIN_PCT)

    tgt_context_sets += get_pointwise_tgt_context_sets(
        data.train_returns_df.select(data.tickers),
        context_size=CONTEXT_SIZE,
        verbose=True,
        iqr_noise_reduction=True,
    )
    print("="*20)
print(f"Total number of target context sets: {len(tgt_context_sets)}")

Running for period 1


No change made because period entered is 1


  0%|          | 0/611 [00:00<?, ?it/s]

  2%|▏         | 13/611 [00:00<00:04, 124.75it/s]

  5%|▍         | 28/611 [00:00<00:04, 127.46it/s]

...

Running for period 10


  0%|          | 0/611 [00:00<?, ?it/s]

 90%|████████▉ | 547/611 [00:00<00:00, 5449.88it/s]

100%|██████████| 611/611 [00:00<00:00, 4795.69it/s]

Total number of target context sets: 1860195


In [2]:
import polars as pl
import numpy as np

ticker = "JPM"
i = data.ticker2idx[ticker]
temp = np.array([xi[1] for xi in tgt_context_sets if xi[0] == i]).flatten()
top_values = pl.Series("value", temp).value_counts(sort=True)["value"].to_list()
print(
    f"The most commonly co-occurring stocks with {ticker} are: {[data.idx2ticker[xi] for xi in top_values][:5]}"
)

The most commonly co-occurring stocks with JPM are: ['C', 'BAC', 'STI', 'WFC', 'PNC']


In [3]:
from stock_embeddings import ClassificationEmbeddings

EMBEDDING_DIM = 20
model = ClassificationEmbeddings(
    n_time_series=len(data.tickers), embedding_dim=EMBEDDING_DIM
)

In [4]:
from stock_embeddings import train_embeddings_from_tgt_context_sets

EPOCHS = 10

model, losses = train_embeddings_from_tgt_context_sets(
    n_time_series=len(data.tickers),
    tgt_context_sets=tgt_context_sets,
    model=model,
    epochs=EPOCHS,
    batch_size=64,
    early_stopping=False,
    device="cpu",
    # embedding_dim=EMBEDDING_DIM,
    verbose=True,
)

Training embeddings...


  0%|          | 0/10 [00:00<?, ?it/s]

 10%|█         | 1/10 [00:17<02:33, 17.09s/it]

Epoch 0: Loss = 0.1037550541554489


 20%|██        | 2/10 [00:34<02:16, 17.12s/it]

...

 90%|█████████ | 9/10 [02:33<00:17, 17.07s/it]

Epoch 8: Loss = 0.0973971480498231


100%|██████████| 10/10 [02:50<00:00, 17.07s/it]

100%|██████████| 10/10 [02:50<00:00, 17.07s/it]

Epoch 9: Loss = 0.09731921040184338


In [5]:
from stock_embeddings import get_sector_score

get_sector_score(model.embeddings.weight.detach().numpy(), sectors=data.sectors, top_k_accuracy=True)

/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Precision Score: 0.64
Recall Score: 0.6
F1 Score: 0.61
Accuracy Score: 0.6
Accuracy Score Top-3: 0.86


/Users/rian/Documents/personal-projects/stock-embeddings/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [6]:
from stock_embeddings.plotting import pca_plot_from_embeddings

pca_plot_from_embeddings(
    embedding_matrix=model.embeddings.weight.detach().numpy(),
    sectors=data.sectors,
    tickers=data.tickers,
    industries=data.industries,
    names=data.names,
    dimensions=2,
    reduced=True,
    method="PCA",
    return_df=False,
    rand_state=None,
)

In [7]:
SAVE_MODEL = False
SAVE_PATH_TEMPLATE = "abs_diff_E{epochs}_C{context_size}_D{embedding_dim}_P{periods}_train{train_pct}.pt"
save_path = SAVE_PATH_TEMPLATE.format(
    epochs=EPOCHS,
    context_size=CONTEXT_SIZE,
    embedding_dim=EMBEDDING_DIM,
    periods="-".join(map(str, PERIODS)),
    train_pct=TRAIN_PCT,
)

# Example usage of saving and loading model
# model.save_model("model_test.pt")
# model = ClassificationEmbeddings.load_model(path="model_test.pt")